# 01. 청년몽땅정보통 크롤링 탐색

셀을 위에서부터 하나씩 실행하면서 사이트 구조가 `CLAUDE.md` 2번 내용과 맞는지 확인하는 노트북입니다.
셀렉터는 아직 `selectors.py`가 없으므로 셀1의 상수로 둡니다.

## 셀1. import와 상수

필요한 라이브러리를 불러오고, URL과 CSS 셀렉터 상수를 한곳에 모읍니다. 
확인할 것: import 에러가 없는지 확인

In [2]:
import re
import time
from datetime import datetime
from pathlib import Path
from urllib.parse import urlencode

import pandas as pd
from selenium import webdriver
from selenium.common.exceptions import NoSuchElementException, TimeoutException
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

KEY = "2309130006"
BASE = "https://youth.seoul.go.kr/infoData/sprtInfo"
LIST_URL = f"{BASE}/list.do"
DETAIL_URL = f"{BASE}/view.do"

# 수집 대상 모집상태 (마감 제외)
STATUSES = ["모집중", "모집예정", "상시"]

# 목록 셀렉터
LIST_ITEM = ".category-feed .feed-item"
LIST_OVERLAY = "a.item-overlay"
LIST_CATE = ".content .cate"
LIST_NAME = ".content .name"
LIST_STATE = ".content .state"

# 상세 셀렉터
DETAIL_TITLE = ".cont .tit strong"
DETAIL_CATE = ".cont .cate"
DETAIL_INFO_LI = "ul.info li"
DETAIL_INFO_LABEL = "em"
DETAIL_APPLY = ".btn-group a"
DETAIL_BODY = ".editor-text"


def list_url(page, per_page=8):
    params = [("key", KEY), ("pageIndex", page), ("recordCountPerPage", per_page), ("orderBy", "regYmd desc")]
    params += [("sc_rcritCurentSitu", s) for s in STATUSES]
    return f"{LIST_URL}?{urlencode(params)}"


def detail_url(source_id):
    return f"{DETAIL_URL}?sprtInfoId={source_id}&key={KEY}"


def safe_text(parent, css):
    # 요소가 없으면 None (크래시 금지)
    try:
        return parent.find_element(By.CSS_SELECTOR, css).text.strip()
    except NoSuchElementException:
        print(f"[경고] 요소 없음: {css}")
        return None

## 셀2. 브라우저 띄우고 목록 페이지 접속

`webdriver.Chrome()`으로 목록 URL에 접속하고, 접속 대기열(WebGate)을 지나 목록 항목이 뜰 때까지 `WebDriverWait`로 기다립니다.
확인할 것: 대기열 때문에 TimeoutException이 나는지, 최종 URL과 페이지 제목이 목록 페이지인지.

In [3]:
driver = webdriver.Chrome()
wait = WebDriverWait(driver, 60)  # 대기열 통과까지 넉넉히

driver.get(list_url(1))
wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, LIST_ITEM)))
time.sleep(1)

print("현재 URL:", driver.current_url)
print("페이지 제목:", driver.title)

현재 URL: https://youth.seoul.go.kr/infoData/sprtInfo/list.do?key=2309130006&pageIndex=1&recordCountPerPage=8&orderBy=regYmd+desc&sc_rcritCurentSitu=%EB%AA%A8%EC%A7%91%EC%A4%91&sc_rcritCurentSitu=%EB%AA%A8%EC%A7%91%EC%98%88%EC%A0%95&sc_rcritCurentSitu=%EC%83%81%EC%8B%9C&WG_V=1790833439004
페이지 제목: 청년정책 > 청년정책검색 > 청년지원정보 목록 | 청년몽땅정보통


## 셀3. 첫 페이지 항목 출력

`.category-feed .feed-item`을 찾고, **항목 하나씩** 하위 요소(분야/제목/모집상태/공고 ID)를 읽어 출력합니다.
확인할 것: 항목 개수, `goView('숫자')`에서 ID가 제대로 뽑히는지, 모집상태에 '마감'이 섞이지 않는지, 분야가 빈 항목이 있는지.

In [4]:
items = driver.find_elements(By.CSS_SELECTOR, LIST_ITEM)
print("항목 수:", len(items))


def parse_list_item(item):
    onclick = None
    try:
        onclick = item.find_element(By.CSS_SELECTOR, LIST_OVERLAY).get_attribute("onclick")
    except NoSuchElementException:
        print(f"[경고] 요소 없음: {LIST_OVERLAY}")
    m = re.search(r"goView\('(\d+)'\)", onclick or "")
    return {
        "source_id": m.group(1) if m else None,
        "category": safe_text(item, LIST_CATE),
        "title": safe_text(item, LIST_NAME),
        "status": safe_text(item, LIST_STATE),
    }


first_page = [parse_list_item(it) for it in items]
for row in first_page:
    print(row)

항목 수: 8
{'source_id': '74476', 'category': '금융', 'title': '금융위원회 & 서민금융진흥원 <청년미래적금 2차 모집>', 'status': '모집예정'}
{'source_id': '68721', 'category': '', 'title': '📢 프로그램 게시요청 가이드 📢', 'status': '상시'}
{'source_id': '74498', 'category': '커뮤니티', 'title': "서울청년센터 성북 10월 월간 성북: 취미 'K-POP 댄스' 참여자 모집", 'status': '모집중'}
{'source_id': '74414', 'category': '교육', 'title': "서울청년센터 도봉 <10월 청년자립메이커: 자립특강 'AI 활용 보고서 작성 및 엑셀 데이터 분석'> 참여자 모집", 'status': '모집중'}
{'source_id': '74517', 'category': '커뮤니티', 'title': '서울청년센터 성북 와글와글 성북마을 하반기 소모임형 :1인분 장보기 회의 (청년 1인가구 먹거리 탐험대)', 'status': '모집중'}
{'source_id': '74516', 'category': '', 'title': '성북길빛도서관 <2026 하반기 성북구 야외도서관 - 책, 도심을 물들이다> 안내', 'status': '모집중'}
{'source_id': '74444', 'category': '', 'title': '서울신용보증재단 성북종합지원센터 <성북동길, 마음환기산책>', 'status': '모집중'}
{'source_id': '74446', 'category': '문화/예술', 'title': '서울신용보증재단 성북종합지원센터 <전문도슨트와 함께하는 미술관 도장깨기>', 'status': '모집중'}


## 셀4. 상세 페이지 1건 접속 후 `ul.info` 라벨별 값 출력

셀3의 첫 공고 상세 페이지로 이동해서 분야/제목, `ul.info li`의 `<em>` 라벨별 값, 신청 링크, 본문 앞 200자를 출력합니다.
확인할 것: 라벨(신청기간/진행일정/대상/담당기관)이 순서와 무관하게 잡히는지, 라벨이 없는 공고가 있는지.

In [5]:
first_id = first_page[0]["source_id"]
driver.get(detail_url(first_id))
wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, DETAIL_TITLE)))
time.sleep(1)


def parse_detail():
    info = {}
    for li in driver.find_elements(By.CSS_SELECTOR, DETAIL_INFO_LI):
        try:
            label = li.find_element(By.CSS_SELECTOR, DETAIL_INFO_LABEL).text.strip()
        except NoSuchElementException:
            continue
        info[label] = li.text.replace(label, "", 1).strip()

    apply_url = None
    try:
        apply_url = driver.find_element(By.CSS_SELECTOR, DETAIL_APPLY).get_attribute("href")
    except NoSuchElementException:
        print(f"[경고] 요소 없음: {DETAIL_APPLY}")

    body = None
    try:
        body = driver.find_element(By.CSS_SELECTOR, DETAIL_BODY).text.strip()[:200]
    except NoSuchElementException:
        print(f"[경고] 요소 없음: {DETAIL_BODY}")

    return {
        "category": safe_text(driver, DETAIL_CATE),
        "title": safe_text(driver, DETAIL_TITLE),
        "info": info,
        "apply_url": apply_url,
        "summary": body,
    }


detail = parse_detail()
print("분야:", detail["category"])
print("제목:", detail["title"])
for label, value in detail["info"].items():
    print(f"[{label}] {value!r}")
print("신청 링크:", detail["apply_url"])
print("요약:", detail["summary"])

분야: 금융
제목: 금융위원회 & 서민금융진흥원 <청년미래적금 2차 모집>
[신청기간] '2026-10-07 ~ 2026-10-16 00 : 00'
[진행일정] '2026-10-07 ~ 2026-10-16 0시 0분 ~ 0시 0분'
[대상] ''
[담당기관] '기타 (금융위원회 )'
신청 링크: https://fill4young.kinfa.or.kr/yfs/main
요약: ○ 사업명 : 청년미래적금 2차 모집
○ 신청기간 : 2026. 10. 7. ~ 10. 16.
- 10. 7. : 출생연도 끝자리 홀수
- 10. 8. : 출생연도 끝자리 짝수
- 10. 12. ~ 10. 16. : 출생연도 관계없이 신청
○ 가입대상 : 계좌 개설 기간 기준 만 19~34세 청년
※ 연령 요건과 개인소득·가구소득 등 세부 조건 충족 필요



## 셀5. 신청기간 파싱 결과 출력

셀4의 '신청기간' 원문에서 `YYYY-MM-DD ~ YYYY-MM-DD`를 정규식으로 뽑습니다. 못 찾으면 start/end는 None, 원문은 그대로 둡니다.
확인할 것: 원문(`repr`)에 줄바꿈·시간이 어떻게 섞여 있는지, 파싱 결과가 맞는지. 아래 샘플 문자열로 정규식도 같이 점검합니다.

In [6]:
PERIOD_RE = re.compile(r"(\d{4}-\d{2}-\d{2})\s*~\s*(\d{4}-\d{2}-\d{2})")


def parse_period(text):
    m = PERIOD_RE.search(text or "")
    if not m:
        return None, None
    return m.group(1), m.group(2)


raw = detail["info"].get("신청기간")
print("원문:", repr(raw))
print("파싱:", parse_period(raw))

print("--- 샘플 점검 ---")
for s in ["2026-09-29 ~ 2026-10-09\n 00 : 00", "상시", "", None]:
    print(repr(s), "->", parse_period(s))

원문: '2026-10-07 ~ 2026-10-16 00 : 00'
파싱: ('2026-10-07', '2026-10-16')
--- 샘플 점검 ---
'2026-09-29 ~ 2026-10-09\n 00 : 00' -> ('2026-09-29', '2026-10-09')
'상시' -> (None, None)
'' -> (None, None)
None -> (None, None)


## 셀6. 2페이지까지 돌려 CSV 저장

`pageIndex`를 바꾼 URL로 `driver.get`하며 1~2페이지를 수집하고, 각 공고 상세까지 방문해 `../output/explore_날짜.csv`로 저장합니다.
페이지 이동마다 1초 이상 대기하고, 항목이 0개인 페이지가 나오면 멈춥니다. 상세 방문을 빼고 싶으면 `FETCH_DETAIL = False`.
확인할 것: 행 수, 중복 ID 여부, 신청기간 파싱 실패 건수, CSV를 엑셀에서 열었을 때 한글이 깨지지 않는지.

In [7]:
MAX_PAGES = 2
FETCH_DETAIL = True

rows = []
for page in range(1, MAX_PAGES + 1):
    driver.get(list_url(page))
    try:
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, LIST_ITEM)))
    except TimeoutException:
        print(f"{page}페이지: 항목 없음 -> 종료")
        break
    time.sleep(1)

    page_rows = [parse_list_item(it) for it in driver.find_elements(By.CSS_SELECTOR, LIST_ITEM)]
    print(f"{page}페이지: {len(page_rows)}건")

    for row in page_rows:
        row["page"] = page
        row["source_url"] = detail_url(row["source_id"]) if row["source_id"] else None
        rows.append(row)

if FETCH_DETAIL:
    for row in rows:
        if not row["source_id"]:
            continue
        driver.get(row["source_url"])
        try:
            wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, DETAIL_TITLE)))
        except TimeoutException:
            print(f"[경고] 상세 로딩 실패: {row['source_id']}")
            continue
        time.sleep(1)
        d = parse_detail()
        period_text = d["info"].get("신청기간")
        row["period_text"] = period_text
        row["start_date"], row["end_date"] = parse_period(period_text)
        row["organization"] = d["info"].get("담당기관")
        row["target"] = d["info"].get("대상")
        row["apply_url"] = d["apply_url"]
        row["summary"] = d["summary"]

df = pd.DataFrame(rows)
out_dir = Path("../output")
out_dir.mkdir(exist_ok=True)
out_path = out_dir / f"explore_{datetime.now():%Y%m%d}.csv"
df.to_csv(out_path, index=False, encoding="utf-8-sig")
print("저장:", out_path.resolve(), "/ 행 수:", len(df))
df.head()

1페이지: 8건
2페이지: 8건
저장: C:\Users\AISW_203_113\Documents\GitHub\Seoul-youth-policy-crawler-website\output\explore_20261001.csv / 행 수: 16


,source_id,category,title,status,page,source_url,period_text,start_date,end_date,organization,target,apply_url,summary
0,74476,금융,금융위원회 & 서민금융진흥원 <청년미래적금 2차 모집>,모집예정,1,https://youth.seoul.go.kr/infoData/sprtInfo/vi...,2026-10-07 ~ 2026-10-16 00 : 00,2026-10-07,2026-10-16,기타 (금융위원회 ),,https://fill4young.kinfa.or.kr/yfs/main,○ 사업명 : 청년미래적금 2차 모집\n○ 신청기간 : 2026. 10. 7. ~ ...
1,68721,,📢 프로그램 게시요청 가이드 📢,상시,1,https://youth.seoul.go.kr/infoData/sprtInfo/vi...,상시,NaN,NaN,서울광역청년센터,,https://youth.seoul.go.kr/infoData/sprtInfo/li...,"서울시 청년들에게 필요한 시의성, 공공성 있는 정보를 게시합니다.\n\n* 프로그램..."
2,74498,커뮤니티,서울청년센터 성북 10월 월간 성북: 취미 'K-POP 댄스' 참여자 모집,모집중,1,https://youth.seoul.go.kr/infoData/sprtInfo/vi...,2026-09-30 ~ 2026-10-13 23 : 59,2026-09-30,2026-10-13,서울청년센터,제한없음,https://yeyak.seoul.go.kr/web/reservation/sele...,"💃10월 월간 성북: 취미 'K-POP 댄스' 🕺\n\n\n춤, 한 번쯤 배워보고 ..."
3,74414,교육,서울청년센터 도봉 <10월 청년자립메이커: 자립특강 'AI 활용 보고서 작성 및 엑...,모집중,1,https://youth.seoul.go.kr/infoData/sprtInfo/vi...,2026-09-23 ~ 2026-10-26,2026-09-23,2026-10-26,서울청년센터,,https://youth.seoul.go.kr/orang/cntr/intro.do?...,[일정 변경으로 인한 재모집]\n\n내부 사정으로 인해 프로그램 일정 및 내용이 변...
4,74517,커뮤니티,서울청년센터 성북 와글와글 성북마을 하반기 소모임형 :1인분 장보기 회의 (청년 1...,모집중,1,https://youth.seoul.go.kr/infoData/sprtInfo/vi...,2026-10-01 ~ 2026-10-11 23 : 59,2026-10-01,2026-10-11,서울청년센터,제한없음,https://forms.gle/Tmc4HVt2PkKd7sjx9,[서울청년센터 성북] 와글와글 성북마을 하반기 소모임형 커뮤니티(청년 1인가구 먹거...


## 셀7. 브라우저 종료

확인이 끝나면 실행해서 Chrome을 닫습니다.

In [8]:
driver.quit()